# 🧠 FashionMNIST - Deep Learning Image Classification

Notebook này hướng dẫn toàn bộ quy trình xây dựng mô hình phân loại ảnh FashionMNIST.

**Nhóm thực hiện:** Thiện, Bảo, Bình, Khoa, Việt, Hưng

---

> ⚡ **Lưu ý:** Vào `Runtime > Change runtime type > T4 GPU` để tăng tốc huấn luyện.

## 0. Clone repo từ GitHub & Setup

In [ ]:
# === Clone repository từ GitHub ===
# Thay <USERNAME> và <REPO_NAME> bằng thông tin thực tế
!git clone https://github.com/<USERNAME>/<REPO_NAME>.git
%cd <REPO_NAME>

In [ ]:
# === Cài đặt dependencies ===
# (PyTorch đã có sẵn trên Colab, chỉ cần cài thêm nếu thiếu)
!pip install -q torchvision matplotlib seaborn scikit-learn tqdm

In [ ]:
# === Kiểm tra GPU ===
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

# Tên 10 class trong FashionMNIST
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

## 2. Tải & Tiền xử lý dữ liệu (Bảo)

In [ ]:
# === Định nghĩa Transform ===
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

# === Tải dữ liệu ===
full_train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=test_transform
)

# === Chia train / validation (80/20) ===
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

print(f'Train size: {len(train_dataset)}')
print(f'Validation size: {len(val_dataset)}')
print(f'Test size: {len(test_dataset)}')

# === Tạo DataLoader ===
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

### 2.1 Trực quan hóa dữ liệu

In [ ]:
# === Hiển thị mẫu ảnh từ mỗi class ===
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Mẫu ảnh từ 10 class FashionMNIST', fontsize=16)

for i in range(10):
    # Tìm 1 ảnh thuộc class i
    idx = next(j for j, (_, label) in enumerate(test_dataset) if label == i)
    img, label = test_dataset[idx]
    ax = axes[i // 5, i % 5]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Xây dựng mô hình (Thiện + Bình)

In [ ]:
class FashionCNN(nn.Module):
    """Mạng CNN cho phân loại FashionMNIST."""

    def __init__(self, num_classes=10):
        super(FashionCNN, self).__init__()

        # Block 1: Conv -> BatchNorm -> ReLU -> MaxPool
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)    # (1,28,28) -> (32,28,28)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)                            # (32,28,28) -> (32,14,14)

        # Block 2: Conv -> BatchNorm -> ReLU -> MaxPool
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)   # (32,14,14) -> (64,14,14)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)                            # (64,14,14) -> (64,7,7)

        # Block 3: Conv -> BatchNorm -> ReLU
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # (64,7,7) -> (128,7,7)
        self.bn3 = nn.BatchNorm2d(128)

        # Classifier
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.flatten(x)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# === Khởi tạo model ===
model = FashionCNN(num_classes=10).to(device)

# === In thông tin model ===
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: FashionCNN')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(model)

## 4. Huấn luyện mô hình (Bình)

In [ ]:
# === Cấu hình huấn luyện ===
EPOCHS = 30
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# === Lưu lịch sử training ===
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Huấn luyện 1 epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Đánh giá trên tập validation/test."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

In [ ]:
# === Training Loop ===
best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Cập nhật scheduler
    scheduler.step(val_loss)

    # Lưu lịch sử
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Lưu best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        marker = ' ✅ Best!'
    else:
        marker = ''

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch+1:2d}/{EPOCHS}] '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | '
          f'LR: {current_lr:.6f}{marker}')

print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.2f}%')

### 4.1 Biểu đồ Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 5. Đánh giá mô hình (Khoa)

In [ ]:
# === Load best model ===
model.load_state_dict(torch.load('best_model.pth'))

# === Đánh giá trên tập test ===
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f'📊 Test Loss: {test_loss:.4f}')
print(f'📊 Test Accuracy: {test_acc:.2f}%')

In [ ]:
# === Thu thập predictions ===
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# === Classification Report ===
print('\n📋 Classification Report:')
print('=' * 60)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# === Confusion Matrix ===
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - FashionMNIST')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 5.1 Hiển thị mẫu dự đoán

In [ ]:
# === Hiển thị một số mẫu dự đoán ===
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Mẫu dự đoán (Xanh = Đúng, Đỏ = Sai)', fontsize=16)

indices = np.random.choice(len(test_dataset), 15, replace=False)

for i, idx in enumerate(indices):
    img, true_label = test_dataset[idx]
    ax = axes[i // 5, i % 5]

    # Predict
    model.eval()
    with torch.no_grad():
        output = model(img.unsqueeze(0).to(device))
        prob = F.softmax(output, dim=1)
        confidence, pred_label = prob.max(1)

    pred_label = pred_label.item()
    confidence = confidence.item() * 100

    # Hiển thị
    ax.imshow(img.squeeze(), cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {CLASS_NAMES[pred_label]}\n'
                 f'True: {CLASS_NAMES[true_label]}\n'
                 f'Conf: {confidence:.1f}%',
                 color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Lưu & Tải model (Việt)

In [ ]:
# === Lưu model hoàn chỉnh ===
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': EPOCHS,
    'best_val_acc': best_val_acc,
    'history': history,
    'class_names': CLASS_NAMES,
}, 'fashion_mnist_full_checkpoint.pth')

print('✅ Model saved successfully!')

# === Download model về máy (trên Colab) ===
from google.colab import files
files.download('best_model.pth')

In [ ]:
# === Export sang ONNX (tùy chọn) ===
dummy_input = torch.randn(1, 1, 28, 28).to(device)
torch.onnx.export(
    model, dummy_input, 'fashion_mnist.onnx',
    input_names=['image'],
    output_names=['prediction'],
    dynamic_axes={'image': {0: 'batch_size'}, 'prediction': {0: 'batch_size'}}
)
print('✅ ONNX model exported!')

## 7. Kết luận

### Tóm tắt kết quả:
- **Model:** FashionCNN (3 Conv blocks + 2 FC layers)
- **Best Validation Accuracy:** ___(điền sau khi train)___
- **Test Accuracy:** ___(điền sau khi train)___

### Hướng phát triển:
- Thử nghiệm kiến trúc nâng cao (ResNet, EfficientNet)
- Hyperparameter tuning với Optuna
- Triển khai API inference
- Xây dựng demo UI